# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnshGarg13622/FlyRank_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is primarily a **ranking/scoring task**. The goal is to assign each page an opportunity score and produce a ranked list of pages that should be reviewed first for potential CTR or engagement improvement.

Ranking is more suitable than simple classification because the main decision is not just whether a page is an opportunity or not, but **which pages deserve attention first** when reviewer capacity is limited. The ranking can use signals such as impressions, CTR, average position, sessions, engagement rate, and scroll rate. I will evaluate the usefulness of the ranking using metrics such as **Precision@K**, while comparing it with a transparent rule-based baseline.


In [ ]:
task_type = "ranking / scoring"

print("Task type:", task_type)
print("Output: a priority score and ranked review queue for content pages")

Task type: ranking / scoring
Output: a priority score and ranked review queue for content pages


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter dataset provides an observed `trend_direction` field. For an initial supervised experiment, I can use a defined proxy target:

**Proxy target:** `is_declining_label = (trend_direction == "down")`.

This proxy represents whether a page is currently observed as declining in the available window. It is useful for learning the framing, but it is not the same as a future causal outcome. A stronger capstone target would ideally be a future observed outcome, such as whether a page's performance declines in a later period.

The final output would be a page-level priority score/ranking, which can support a human review queue.

In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    !git clone -q https://github.com/AnshGarg13622/FlyRank_Internship.git /content/FlyRank_Internship
    %cd /content/FlyRank_Internship
    data_path = Path("data/raw/content_refresh_anonymized.csv")

# Load starter dataset
df = pd.read_csv(data_path)

# Check available columns related to the target
target_cols = [
    "ctr",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "trend_direction"
]

print("Available target-related columns:")
for col in target_cols:
    print(f"{col}: {'Yes' if col in df.columns else 'No'}")

# Show the distribution of the available starter proxy
if "trend_direction" in df.columns:
    print("\nTrend direction distribution:")
    print(df["trend_direction"].value_counts(dropna=False))

/content/FlyRank_Internship
Available target-related columns:
ctr: Yes
impressions_90d: Yes
clicks_90d: Yes
sessions_90d: Yes
trend_direction: Yes

Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*


The primary success metric will be **Precision@K**, with **K = 50** as the initial review capacity. Precision@50 measures the proportion of the top 50 ranked pages that match the defined review-opportunity target or proxy.

I will consider the ranking useful if it achieves **at least 50% Precision@50**, meaning that at least half of the first 50 recommended pages meet the defined opportunity criterion. This threshold is a practical benchmark rather than a universal standard, and the final result will be compared against the transparent baseline.

Precision@50 is appropriate because the business decision is about **which limited set of pages should be reviewed first**, rather than predicting an outcome for every page.

In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create a simple proxy for CTR review opportunity
# Only consider pages with enough impressions
eval_df = df[
    (df["impressions_90d"] >= 500) &
    (df["ctr"].notna())
].copy()

# Define proxy label for baseline evaluation
eval_df["opportunity_label"] = (
    eval_df["ctr"] < 0.5
).astype(int)

# Create a simple baseline score:
# Higher impressions + lower CTR = higher priority
eval_df["baseline_score"] = (
    eval_df["impressions_90d"].rank(pct=True) *
    (1 - eval_df["ctr"].rank(pct=True))
)

# Rank pages
eval_df = eval_df.sort_values(
    "baseline_score",
    ascending=False
)

# Calculate Precision@K
K = min(50, len(eval_df))

precision_at_k = eval_df.head(K)["opportunity_label"].mean()

print(f"Evaluation pages: {len(eval_df):,}")
print(f"Precision@{K}: {precision_at_k:.3f}")
print(f"Precision@{K} (%): {precision_at_k * 100:.1f}%")

Evaluation pages: 16,726
Precision@50: 1.000
Precision@50 (%): 100.0%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

For the starter dataset, the **unit of analysis is one content page**. Each row represents a unique `content_id` with aggregated 90-day search and engagement signals such as impressions, clicks, CTR, average position, sessions, and engagement metrics.

The Lane 4 slice is therefore structured as **one row = one content page**, allowing pages to be compared and ranked for potential CTR or engagement review. I will verify that `content_id` is unique after filtering so that the same page is not counted multiple times.


In [3]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Lane 4 slice:
# Keep pages with measurable search visibility
lane4_df = df[
    (df["impressions_90d"] > 0)
].copy()

# Remove duplicate content pages if present
lane4_df = lane4_df.drop_duplicates(subset=["content_id"])

print("Lane 4 dataframe shape:", lane4_df.shape)

print("\nUnit of analysis:")
print("One row = one content page")

print("\nColumns:")
print(lane4_df.columns.tolist())

print("\nSample rows:")
display(lane4_df.head(10))

# Verify whether content_id is unique
print("\nUnique content IDs:", lane4_df["content_id"].nunique())
print("Total rows:", len(lane4_df))

if lane4_df["content_id"].nunique() == len(lane4_df):
    print("✓ Each row represents one unique content page.")
else:
    print("⚠ Duplicate content IDs still exist.")

Lane 4 dataframe shape: (30000, 44)

Unit of analysis:
One row = one content page

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Sample rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,1.00,HIGH,1.05,keyword article,transactional,3080.0,18178.0,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,LOW,0.00,keyword article,informational,3059.0,20810.0,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.44,MEDIUM,0.64,keyword article,commercial,NaN,NaN,...,NaN,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3807.0,24228.0,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,NaN,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2



Unique content IDs: 30000
Total rows: 30000
✓ Each row represents one unique content page.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as **“CTR below 0.5% = opportunity”** is too simplistic because CTR depends on several interacting factors. In particular, pages in different search-position ranges naturally have different click-through behaviour, while impressions, content type, search intent, freshness, and engagement context can also affect the observed CTR.

Therefore, the same CTR value does not necessarily represent the same level of opportunity for every page. A data-driven scoring approach can combine multiple signals and rank pages based on their overall context rather than relying on a single threshold.

ML is justified only if it demonstrates better ranking performance than the transparent rule-based baseline. The goal is not to replace a simple rule automatically, but to test whether a more flexible scoring approach provides a more useful review queue.


In [6]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

lane4_df = df[
    (df["impressions_90d"] > 0) &
    (df["ctr"].notna()) &
    (df["avg_position"].notna()) # Corrected column name from average_position to avg_position
].copy()

# Create position groups
lane4_df["position_tier"] = pd.cut(
    lane4_df["avg_position"], # Corrected column name from average_position to avg_position
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

# Compare CTR across position tiers
position_summary = lane4_df.groupby(
    "position_tier",
    observed=True
).agg(
    pages=("content_id", "nunique"),
    median_ctr=("ctr", "median"),
    median_impressions=("impressions_90d", "median")
).reset_index()

display(position_summary)

,position_tier,pages,median_ctr,median_impressions
0,1-3,1141,0.00,74.0
1,4-10,11842,0.16,1184.0
2,11-20,7273,0.10,870.0
3,21-50,7225,0.03,807.0
4,50+,1314,0.00,219.5


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.